# SD3.5 CityPersons Augmentation - Refactored V3
Optimized pipeline with scale correction and appearance harmonization. ~700 LOC, 100% logic preserved.

In [ ]:
# 1. Install Dependencies (Run once)
!pip install -q "diffusers>=0.30.0,<1.0.0" "transformers>=4.40.0" "accelerate>=0.30.0" sentencepiece protobuf safetensors ultralytics

# 2. Imports
import os, re, json, math, random, csv, gc
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Tuple, Any
import numpy as np
import torch
from PIL import Image, ImageOps, ImageDraw, ImageFilter, ImageChops
import matplotlib.pyplot as plt

os.environ["DIFFUSERS_VERBOSITY"] = "error"
import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="diffusers.")

try:
    import cv2
except ImportError:
    cv2 = None

from diffusers import (
    StableDiffusion3Img2ImgPipeline, StableDiffusion3InpaintPipeline,
    StableDiffusionXLImg2ImgPipeline, StableDiffusionXLInpaintPipeline
)

In [ ]:
@dataclass
class AugConfig:
    # Model & Device
    backend: str = "sd35"
    sd35_model_id: str = "stabilityai/stable-diffusion-3.5-medium"
    sdxl_model_id: str = "stabilityai/stable-diffusion-xl-base-1.0"
    device: str = "cuda:0"
    use_cpu_offload: bool = True
    use_t5: bool = False
    resolution: int = 448

    # Dataset & Output
    dataset_root_candidates: List[Path] = field(default_factory=lambda: [
        Path('/kaggle/input/datasets/muttahirulislam/citypersons-dataset-with-bg-image/yolo_dir/yolo_dir'),
        Path('/kaggle/input/citypersons-dataset-with-bg-image')
    ])
    output_dir: Path = Path('/kaggle/working/sd35_citypersons_scale_corrected')
    metrics_dirname: str = "metrics"
    max_train_images: int = 500
    target_splits: List[str] = field(default_factory=lambda: ['train', 'val'])

    # Generation Params
    variants: List[str] = field(default_factory=lambda: [
        'add_single_pedestrian', 'add_two_pedestrians', 'add_small_group', 
        'add_occluded_pedestrian', 'add_distant_pedestrian', 'add_near_pedestrian'
    ])
    augmentations_per_bucket: int = 200
    background_preservation_mode: str = 'context_person_composite'
    max_retries: int = 3

    # Scale & Placement
    patch_road_y_range: Tuple[float, float] = (0.66, 0.92)
    min_person_conf: float = 0.12
    yolo_model_id: str = "yolov8m-seg.pt"
    fallback_diff_threshold: float = 10.0
    fallback_min_change_ratio: float = 0.015
    patch_max_placement_tries: int = 96
    insertion_edge_margin: int = 14
    min_accepted_height_ratio: float = 0.075
    scale_correction_soft_min: float = 0.55
    scale_correction_soft_max: float = 2.2

    # Harmonization
    harmonization_strength: float = 0.75
    add_sensor_noise: bool = True
    noise_std_max: float = 3.0
    use_seamless_clone: bool = True

CFG = AugConfig()
CFG.output_dir.mkdir(parents=True, exist_ok=True)

# Prompt Templates
BASE_PROMPT = "CityPersons traffic-camera street photo. Add {variant_desc}. Keep the original scene unchanged. Full body, feet grounded, plausible scale, matching lighting and perspective."
NEGATIVE_PROMPT = "cropped body, cut off by image border, missing head, missing legs, only legs, only torso, giant person, oversized foreground person, extreme close-up, floating, person on wall, person on building, person on vehicle, ghost, hard seam, transparent person, faded body, blurry"

VARIANT_DESCS = {
    "add_single_pedestrian": "one full-body pedestrian",
    "add_two_pedestrians": "two full-body pedestrians",
    "add_small_group": "three full-body pedestrians",
    "add_occluded_pedestrian": "partly occluded full-body pedestrian",
    "add_distant_pedestrian": "distant but clearly visible full-body pedestrian, not tiny",
    "add_near_pedestrian": "near full-body pedestrian, about 1.5x larger than a normal mid-ground pedestrian, plausible street perspective"
}

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}, CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(token=UserSecretsClient().get_secret("HF_TOKEN"))
    print("Logged in to Hugging Face.")
except Exception as e:
    print("HF login skipped. Add Kaggle secret 'HF_TOKEN' if model is gated.")

In [ ]:
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

@dataclass
class ImageRecord:
    path: Path
    split: str
    bucket: str = "urban_pedestrian_scene"
    caption: str = "CityPersons traffic-camera urban street photo"
    label_path: Optional[Path] = None

def resolve_dataset_root(candidates):
    for path in candidates:
        if (path / "train" / "images").exists() and ((path / "valid" / "images").exists() or (path / "val" / "images").exists()):
            return path
    return Path("/kaggle/input/citypersons-dataset-with-bg-image") # Fallback

DATASET_ROOT = resolve_dataset_root(CFG.dataset_root_candidates)
VALID_SPLIT = "valid" if (DATASET_ROOT / "valid" / "images").exists() else "val"

def scan_dataset(max_images=CFG.max_train_images):
    records = []
    for split, split_name in [("train", "train"), ("val", VALID_SPLIT)]:
        img_dir = DATASET_ROOT / split_name / "images"
        if not img_dir.exists(): continue
        for img_path in sorted(img_dir.rglob("*")):
            if img_path.suffix.lower() in {".jpg", ".jpeg", ".png"} and "mask" not in img_path.name.lower():
                label_path = (DATASET_ROOT / split_name / "labels" / img_path.stem).with_suffix(".txt")
                records.append(ImageRecord(path=img_path, split=split, label_path=label_path if label_path.exists() else None))
                if max_images and len(records) >= max_images: return records
    return records

records = scan_dataset()
print(f"Scanned {len(records)} images from {DATASET_ROOT}")

In [ ]:
PIPELINE_CACHE = {}

def build_pipeline(mode: str = "img2img", device: str = CFG.device):
    cache_key = f"{CFG.backend}_{mode}_{device}"
    if cache_key in PIPELINE_CACHE:
        return PIPELINE_CACHE[cache_key]

    is_sd3 = CFG.backend == "sd35"
    is_inpaint = "inpaint" in mode.lower()
    
    if is_sd3:
        cls = StableDiffusion3InpaintPipeline if is_inpaint else StableDiffusion3Img2ImgPipeline
        model_id = CFG.sd35_model_id
    else:
        cls = StableDiffusionXLInpaintPipeline if is_inpaint else StableDiffusionXLImg2ImgPipeline
        model_id = CFG.sdxl_model_id

    kwargs = {"torch_dtype": torch.float16, "use_safetensors": True, "low_cpu_mem_usage": True}
    if is_sd3 and not CFG.use_t5:
        kwargs.update({"text_encoder_3": None, "tokenizer_3": None})

    if str(device).startswith("cuda"):
        torch.cuda.set_device(torch.device(device).index or 0)

    try:
        pipe = cls.from_pretrained(model_id, **kwargs)
    except TypeError:
        kwargs.pop("text_encoder_3", None)
        kwargs.pop("tokenizer_3", None)
        kwargs.pop("low_cpu_mem_usage", None)
        pipe = cls.from_pretrained(model_id, **kwargs)
    
    if str(device).startswith("cuda") and CFG.use_cpu_offload and hasattr(pipe, "enable_model_cpu_offload"):
        pipe.enable_model_cpu_offload(gpu_id=torch.device(device).index or 0)
    else:
        pipe.to(device)
        
    for opt in ["enable_vae_slicing", "enable_vae_tiling", "enable_attention_slicing"]:
        if hasattr(pipe, opt): getattr(pipe, opt)()

    PIPELINE_CACHE[cache_key] = pipe
    return pipe

In [ ]:
def clamp_bbox(bbox, w, h):
    x1, y1, x2, y2 = bbox
    return (max(0, min(w-1, int(round(x1)))), max(0, min(h-1, int(round(y1)))), 
            max(int(round(x1))+1, min(w, int(round(x2)))), max(int(round(y1))+1, min(h, int(round(y2)))))

def bbox_intersection_area(a, b):
    return max(0, min(a[2], b[2]) - max(a[0], b[0])) * max(0, min(a[3], b[3]) - max(a[1], b[1]))

def bbox_area(bbox):
    return max(0, bbox[2] - bbox[0]) * max(0, bbox[3] - bbox[1])

def load_source_image(path):
    return ImageOps.exif_transpose(Image.open(path)).convert("RGB")

def load_yolo_bboxes(record, original_size, class_ids=(0,), resolution=CFG.resolution):
    if not record.label_path or not Path(record.label_path).exists():
        return []
    ow, oh = original_size
    boxes = []
    for line in Path(record.label_path).read_text(encoding="utf-8").splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        try:
            cls = int(float(parts[0]))
            xc, yc, bw, bh = map(float, parts[1:5])
        except ValueError:
            continue
        if cls not in class_ids:
            continue
        boxes.append(clamp_bbox(((xc - bw / 2) * resolution, (yc - bh / 2) * resolution, (xc + bw / 2) * resolution, (yc + bh / 2) * resolution), resolution, resolution))
    return boxes

def expected_person_height(foot_y, img_h, variant="add_single_pedestrian"):
    ratio = np.clip(0.065 + ((foot_y / max(1.0, img_h)) - 0.55) * 0.58, 0.075, 0.29)
    multiplier = 0.82 if "distant" in variant else (1.13 if "near" in variant else 1.0)
    return max(28.0, ratio * multiplier * img_h)

def choose_insertion_bbox(record, source, variant, rng):
    w = h = CFG.resolution
    old_boxes = load_yolo_bboxes(record, source.size, class_ids=(0, 2, 5, 7), resolution=w)
    best, best_score = None, -1e9
    for _ in range(CFG.patch_max_placement_tries):
        foot_y = rng.randint(int(h * CFG.patch_road_y_range[0]), int(h * CFG.patch_road_y_range[1]))
        person_h = expected_person_height(foot_y, h, variant)
        person_w = person_h * (0.34 if "distant" in variant else 0.38)
        if "two" in variant:
            person_w *= 1.9
        elif "group" in variant:
            person_w *= 2.5
        cx = rng.randint(CFG.insertion_edge_margin, w - CFG.insertion_edge_margin)
        bbox = clamp_bbox((cx - person_w / 2, foot_y - person_h, cx + person_w / 2, foot_y), w, h)
        overlap = sum(bbox_intersection_area(bbox, b) / max(1, bbox_area(bbox)) for b in old_boxes)
        edge_penalty = 0.15 if bbox[0] <= CFG.insertion_edge_margin or bbox[2] >= w - CFG.insertion_edge_margin else 0.0
        score = -overlap - edge_penalty + rng.random() * 0.05
        if score > best_score:
            best, best_score = bbox, score
    return best or (170, 210, 220, 310)

def make_inpaint_mask(bbox, size):
    mask = Image.new("L", size, 0)
    x1, y1, x2, y2 = bbox
    pad = 10
    ImageDraw.Draw(mask).rounded_rectangle(clamp_bbox((x1 - pad, y1 - pad, x2 + pad, y2 + pad), *size), radius=8, fill=255)
    return mask.filter(ImageFilter.GaussianBlur(2))

def generated_image_path(output_dir, record, variant, index):
    safe_variant = variant.replace("/", "_")
    return Path(output_dir) / record.split / record.bucket / "images" / f"{record.path.stem}_aug_{index:04d}_{safe_variant}.png"

def comparison_image_path(output_dir, record, variant, index):
    safe_variant = variant.replace("/", "_")
    return Path(output_dir) / "comparison_pairs" / record.split / record.bucket / f"{record.path.stem}_pair_{index:04d}_{safe_variant}.png"

def save_comparison_pair(original, augmented, comparison_path, title):
    comparison_path = Path(comparison_path)
    comparison_path.parent.mkdir(parents=True, exist_ok=True)
    original = original.convert("RGB").resize((CFG.resolution, CFG.resolution))
    augmented = augmented.convert("RGB").resize(original.size)
    title_h, label_h = 34, 28
    canvas = Image.new("RGB", (original.width * 2, original.height + title_h + label_h), "white")
    draw = ImageDraw.Draw(canvas)
    draw.text((10, 8), title, fill=(0, 0, 0))
    draw.text((10, title_h + 6), "original", fill=(0, 0, 0))
    draw.text((original.width + 10, title_h + 6), "augmented", fill=(0, 0, 0))
    canvas.paste(original, (0, title_h + label_h))
    canvas.paste(augmented, (original.width, title_h + label_h))
    canvas.save(comparison_path)
    return comparison_path

PERSON_SEGMENTER = None
def load_person_segmenter():
    global PERSON_SEGMENTER
    if PERSON_SEGMENTER is False:
        return None
    if PERSON_SEGMENTER is None:
        try:
            from ultralytics import YOLO
            PERSON_SEGMENTER = YOLO(CFG.yolo_model_id)
        except Exception as e:
            print(f"YOLO unavailable, using inpaint mask fallback: {e}")
            PERSON_SEGMENTER = False
            return None
    return PERSON_SEGMENTER

def changed_region_detection(image, background_image, insert_bbox, force=False):
    if background_image is None:
        return []
    image = image.convert("RGB")
    background = background_image.convert("RGB").resize(image.size)
    diff = np.mean(np.abs(np.asarray(image, dtype=np.float32) - np.asarray(background, dtype=np.float32)), axis=2)
    roi = Image.new("L", image.size, 0)
    ImageDraw.Draw(roi).rectangle(insert_bbox, fill=255)
    arr = ((diff >= CFG.fallback_diff_threshold) & (np.asarray(roi) > 0)).astype(np.uint8) * 255
    if cv2 is not None:
        kernel = np.ones((5, 5), np.uint8)
        arr = cv2.morphologyEx(arr, cv2.MORPH_CLOSE, kernel, iterations=2)
        arr = cv2.dilate(arr, kernel, iterations=1)
    changed_ratio = float((arr > 0).mean())
    if changed_ratio < CFG.fallback_min_change_ratio and not force:
        return []
    if changed_ratio < CFG.fallback_min_change_ratio:
        arr = np.asarray(roi, dtype=np.uint8)
    ys, xs = np.where(arr > 0)
    if len(xs) == 0 or len(ys) == 0:
        return []
    bbox = clamp_bbox((xs.min(), ys.min(), xs.max() + 1, ys.max() + 1), *image.size)
    mask = clean_binary_person_mask(Image.fromarray(arr, mode="L"), keep_components=2)
    return [{"bbox": bbox, "mask": mask, "conf": 0.01, "fallback": True, "changed_ratio": changed_ratio}]

def detect_new_persons(image, insert_bbox, device, background_image=None, force_fallback=False):
    model = load_person_segmenter()
    if model is None:
        fallback = changed_region_detection(image, background_image, insert_bbox, force=True)
        if fallback:
            return fallback
        mask = Image.new("L", image.size, 0)
        ImageDraw.Draw(mask).rectangle(insert_bbox, fill=255)
        return [{"bbox": insert_bbox, "mask": mask, "conf": 0.01, "fallback": True}]
    try:
        results = model.predict(np.asarray(image), classes=[0], conf=CFG.min_person_conf, imgsz=CFG.resolution, device=str(device), verbose=False)
    except Exception as e:
        print(f"YOLO detection failed, using inpaint mask fallback: {e}")
        fallback = changed_region_detection(image, background_image, insert_bbox, force=True)
        if fallback:
            return fallback
        mask = Image.new("L", image.size, 0)
        ImageDraw.Draw(mask).rectangle(insert_bbox, fill=255)
        return [{"bbox": insert_bbox, "mask": mask, "conf": 0.01, "fallback": True}]
    detections = []
    if not results:
        return detections
    result = results[0]
    boxes = result.boxes.xyxy.cpu().numpy() if result.boxes is not None else []
    confs = result.boxes.conf.cpu().numpy() if result.boxes is not None else []
    masks = result.masks.data.cpu().numpy() if result.masks is not None else [None] * len(boxes)
    for box, conf, raw_mask in zip(boxes, confs, masks):
        bbox = clamp_bbox(box, *image.size)
        if bbox_intersection_area(bbox, insert_bbox) / max(1, bbox_area(bbox)) < 0.25:
            continue
        mask = Image.fromarray((raw_mask * 255).astype(np.uint8), mode="L").resize(image.size, Image.NEAREST) if raw_mask is not None else Image.new("L", image.size, 0)
        if raw_mask is None:
            ImageDraw.Draw(mask).rectangle(bbox, fill=255)
        detections.append({"bbox": bbox, "mask": mask, "conf": float(conf)})
    if detections:
        return sorted(detections, key=lambda d: d["conf"], reverse=True)[:3]
    return changed_region_detection(image, background_image, insert_bbox, force=force_fallback)

def clean_binary_person_mask(mask: Image.Image, keep_components: int = 1) -> Image.Image:
    if cv2 is None: return mask
    arr = (np.asarray(mask.convert("L"), dtype=np.uint8) >= 96).astype(np.uint8) * 255
    kernel = np.ones((3, 3), np.uint8)
    arr = cv2.morphologyEx(arr, cv2.MORPH_CLOSE, kernel, iterations=2)
    
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats((arr > 0).astype(np.uint8), 8)
    if num_labels > 1:
        component_ids = sorted(range(1, num_labels), key=lambda i: stats[i, cv2.CC_STAT_AREA], reverse=True)
        largest_area = float(stats[component_ids[0], cv2.CC_STAT_AREA]) if component_ids else 0.0
        keep = {cid for cid in component_ids[:max(1, keep_components)] if stats[cid, cv2.CC_STAT_AREA] >= max(8.0, largest_area * 0.012)}
        if keep: arr = np.where(np.isin(labels, list(keep)), 255, 0).astype(np.uint8)
        
    # Fill holes
    ys, xs = np.where(arr > 0)
    if len(xs) > 0 and len(ys) > 0:
        x1, x2, y1, y2 = int(xs.min()), int(xs.max()) + 1, int(ys.min()), int(ys.max()) + 1
        roi = arr[y1:y2, x1:x2]
        if roi.shape[0] > 2 and roi.shape[1] > 2:
            padded = np.pad(roi, ((1, 1), (1, 1)), mode="constant", constant_values=0)
            flood = padded.copy()
            flood_mask = np.zeros((flood.shape[0] + 2, flood.shape[1] + 2), dtype=np.uint8)
            cv2.floodFill(flood, flood_mask, (0, 0), 255)
            arr[y1:y2, x1:x2] = cv2.bitwise_or(roi, cv2.bitwise_not(flood)[1:-1, 1:-1])
            
    return Image.fromarray(cv2.morphologyEx(arr, cv2.MORPH_CLOSE, kernel, iterations=1), mode="L")

In [ ]:
class Harmonizer:
    @staticmethod
    def apply(source_crop: Image.Image, person_rgb: Image.Image, person_mask: Image.Image, cfg: AugConfig) -> Image.Image:
        """Thực hiện Color, Brightness, Contrast, Noise và Blur trong 1 lần duyệt NumPy duy nhất."""
        src_arr = np.asarray(source_crop.convert("RGB"), dtype=np.float32)
        gen_arr = np.asarray(person_rgb.convert("RGB"), dtype=np.float32)
        mask_arr = np.asarray(person_mask.convert("L"), dtype=np.float32) / 255.0
        
        alpha = np.expand_dims(np.clip(mask_arr, 0.0, 1.0), axis=2)
        core_alpha = np.expand_dims(np.clip((mask_arr > 0.8).astype(float), 0.0, 1.0), axis=2)
        edge_alpha = alpha * cfg.harmonization_strength
        
        # 1. Color & Brightness Transfer
        active_src, active_gen = src_arr[core_alpha[..., 0] > 0.5], gen_arr[core_alpha[..., 0] > 0.5]
        if len(active_src) > 0 and len(active_gen) > 0:
            src_mean, src_std = np.mean(active_src, axis=0), np.std(active_src, axis=0) + 1e-6
            gen_mean, gen_std = np.mean(active_gen, axis=0), np.std(active_gen, axis=0) + 1e-6
            
            ratio = np.clip(src_std / gen_std, 0.8, 1.2)
            corrected = (gen_arr - gen_mean) * ratio + src_mean
            
            # 2. Sensor Noise (nếu cần)
            if cfg.add_sensor_noise:
                src_hf = np.std(src_arr - np.asarray(source_crop.filter(ImageFilter.GaussianBlur(1)), dtype=np.float32))
                gen_hf = np.std(gen_arr - np.asarray(person_rgb.filter(ImageFilter.GaussianBlur(1)), dtype=np.float32))
                if gen_hf < src_hf * 0.92:
                    noise = np.random.normal(0, min(cfg.noise_std_max, (src_hf - gen_hf) * 0.38), gen_arr.shape).astype(np.float32)
                    corrected = corrected + noise * edge_alpha
                    
            # 3. Final Blend
            final_arr = gen_arr * (1.0 - edge_alpha) + corrected * edge_alpha
            return Image.fromarray(np.clip(final_arr, 0, 255).astype(np.uint8), mode="RGB")
        return person_rgb

In [ ]:
from dataclasses import dataclass, field
from typing import Optional, Tuple

@dataclass
class ValidationResult:
    is_valid: bool
    reason: str
    mask: Optional[Image.Image] = None
    bbox: Optional[Tuple] = None
    corrected_image: Optional[Image.Image] = None
    meta: Dict = field(default_factory=dict)

def expected_person_height(foot_y, img_h, variant="add_single_pedestrian"):
    ratio = np.clip(0.065 + ((foot_y / max(1.0, img_h)) - 0.55) * 0.58, 0.075, 0.29)
    multiplier = 0.82 if "distant" in variant else (1.13 if "near" in variant else 1.0)
    return max(28.0, ratio * multiplier * img_h)

def validate_and_correct_scale(generated_image, detections, variant):
    """Kiểm tra và điều chỉnh tỷ lệ người được tạo ra dựa trên phối cảnh."""
    w, h = generated_image.size
    corrected_rgb = Image.new("RGB", (w, h), (0, 0, 0))
    combined_mask = Image.new("L", (w, h), 0)
    corrected_bboxes, per_person_meta = [], []
    
    for det in detections:
        x1, y1, x2, y2 = [int(round(v)) for v in det["bbox"]]
        det_h, det_w = max(1.0, y2 - y1), max(1.0, x2 - x1)
        exp_h = expected_person_height(float(y2), h, variant)
        scale_ratio = exp_h / max(1.0, det_h)
        
        # Policy check
        if scale_ratio < CFG.scale_correction_soft_min or scale_ratio > CFG.scale_correction_soft_max:
            return None, None, None, {"reject_reason": "scale_unrecoverable"}, "scale_unrecoverable"

        # Resize & Paste
        new_h, new_w = int(round(det_h * scale_ratio)), int(round(det_w * scale_ratio))
        new_y2, new_xc = int(round(y2)), (x1 + x2) / 2.0
        new_x1, new_y1 = int(round(new_xc - new_w / 2.0)), new_y2 - new_h
        
        crop = generated_image.crop((x1, y1, x2, y2)).resize((max(4, int(new_w)), max(8, int(new_h))), Image.LANCZOS)
        raw_mask = det["mask"].resize((w, h), Image.NEAREST).crop((x1, y1, x2, y2)).resize(crop.size, Image.NEAREST)
        clean_mask = clean_binary_person_mask(raw_mask)
        px1, py1 = max(0, new_x1), max(0, new_y1)
        px2, py2 = min(w, new_x1 + crop.width), min(h, new_y1 + crop.height)
        if px2 <= px1 or py2 <= py1:
            continue
        sx1, sy1 = px1 - new_x1, py1 - new_y1
        sx2, sy2 = sx1 + (px2 - px1), sy1 + (py2 - py1)
        crop = crop.crop((sx1, sy1, sx2, sy2))
        clean_mask = clean_mask.crop((sx1, sy1, sx2, sy2))
        if np.asarray(clean_mask).mean() < 2.0:
            continue
        corrected_rgb.paste(crop, (px1, py1), clean_mask)
        combined_mask.paste(clean_mask, (px1, py1), clean_mask)
        corrected_bboxes.append((px1, py1, px2, py2))
        per_person_meta.append({"scale_corrected": abs(scale_ratio - 1.0) > 0.05, "scale_ratio": round(float(scale_ratio), 3)})

    if not corrected_bboxes:
        return None, None, None, {"reject_reason": "no_person_detected"}, "no_person_detected"
        
    return corrected_rgb, clean_binary_person_mask(combined_mask), (min(b[0] for b in corrected_bboxes), min(b[1] for b in corrected_bboxes), max(b[2] for b in corrected_bboxes), max(b[3] for b in corrected_bboxes)), {"scale_corrected": any(m["scale_corrected"] for m in per_person_meta), "scale_ratios": json.dumps([m["scale_ratio"] for m in per_person_meta])}, "ok"

In [ ]:
def generate_pedestrian_composite(pipe, source, record, variant, seed, device):
    prompt = BASE_PROMPT.format(variant_desc=VARIANT_DESCS.get(variant, "a pedestrian"))
    crop_source = source.resize((CFG.resolution, CFG.resolution), Image.LANCZOS)
    rng = random.Random(seed)
    last_reason = ""
    
    for attempt in range(CFG.max_retries + 1):
        generator = torch.Generator(device=device).manual_seed(seed + attempt * 9973)
        
        # Adaptive retry params (tăng strength/guidance nếu lỗi)
        strength = 0.72 + (0.04 * attempt)
        guidance = 6.8 + (0.35 * attempt)
        attempt_prompt = prompt
        attempt_negative = NEGATIVE_PROMPT
        if last_reason in {"no_person_detected", "scale_unrecoverable"} or attempt > 0:
            attempt_prompt += ", clearly visible solid full-body pedestrian, distinct natural silhouette, opaque person, sharp body outline"
            attempt_negative += ", invisible person, empty street, no pedestrian, low contrast person, transparent body, faint body"
            strength = min(0.88, strength + 0.04)
            guidance = min(8.6, guidance + 0.45)
        insert_bbox = choose_insertion_bbox(record, source, variant, rng)
        inpaint_mask = make_inpaint_mask(insert_bbox, crop_source.size)
        
        # 1. Generate only inside the proposed insertion region.
        generated = pipe(
            prompt=attempt_prompt, negative_prompt=attempt_negative, image=crop_source, mask_image=inpaint_mask,
            strength=strength, guidance_scale=guidance, num_inference_steps=36, generator=generator
        ).images[0].resize(crop_source.size)
        
        # 2. Detect the new person, then validate and correct scale.
        detections = detect_new_persons(generated, insert_bbox, device, background_image=crop_source, force_fallback=(attempt == CFG.max_retries))
        
        corr_img, corr_mask, corr_bbox, meta, reason = validate_and_correct_scale(generated, detections, variant)
        
        if reason == "ok" and corr_mask is not None:
            # 3. Harmonize
            harmonized = Harmonizer.apply(crop_source, corr_img, corr_mask, CFG)
            
            final_image = crop_source.copy()
            final_image.paste(harmonized, (0, 0), corr_mask)
            meta.update({"insert_bbox": json.dumps(insert_bbox), "attempt": attempt, "pipeline": "inpaint_compact", "fallback_detection_used": any(d.get("fallback") for d in detections)})
            return final_image, corr_bbox, meta
            
        print(f"Retry {attempt+1}/{CFG.max_retries} due to: {reason}")
        last_reason = reason
        
    raise RuntimeError(f"Failed to generate valid person after {CFG.max_retries} retries.")

In [ ]:
def run_augmentation_jobs(devices: List[str], jobs: List[Dict]):
    manifest_rows = []
    for device in devices:
        pipe = build_pipeline(mode="inpaint", device=device)
        for job in jobs:
            try:
                img, bbox, meta = generate_pedestrian_composite(
                    pipe, job["source"], job["record"], job["variant"], job["seed"], device
                )
                output_path = Path(job["output_path"])
                output_path.parent.mkdir(parents=True, exist_ok=True)
                img.save(output_path)
                comparison_path = job.get("comparison_path")
                if comparison_path:
                    save_comparison_pair(job["source"], img, comparison_path, f"{job['variant']} seed={job['seed']}")
                row = {
                    "split": job["record"].split,
                    "bucket": job["record"].bucket,
                    "source_context": job["record"].caption,
                    "source_timeofday": "",
                    "source_scene": job["record"].bucket,
                    "target_insertion": job["variant"].replace("add_", ""),
                    "target_timeofday": "",
                    "original_path": str(job["record"].path),
                    "augmented_path": str(output_path),
                    "comparison_path": str(comparison_path or ""),
                    "variant": job["variant"],
                    "strength": job.get("strength", "adaptive"),
                    "guidance_scale": job.get("guidance_scale", "adaptive"),
                    "num_inference_steps": job.get("num_inference_steps", 36),
                    "generation_mode": meta.get("pipeline", "inpaint_compact"),
                    "insert_bbox": meta.get("insert_bbox", ""),
                    "patch_bbox": meta.get("insert_bbox", ""),
                    "patch_debug_path": "",
                    "person_bbox": json.dumps(bbox),
                    "scale_corrected": meta.get("scale_corrected", False),
                    "scale_ratios": meta.get("scale_ratios", "[]"),
                    "fallback_detection_used": meta.get("fallback_detection_used", False),
                    "scale_correction_status": "corrected" if meta.get("scale_corrected") else "accepted",
                    "seamless_clone_used": False,
                    "fallback_alpha_paste": True,
                    "fallback_alpha_used": True,
                    "retry_attempts": meta.get("attempt", 0),
                    "last_reject_reason": "",
                    "reject_reason": "",
                    "seed": job["seed"],
                    "source_path": str(job["record"].path),
                    "label_path": str(job["record"].label_path or ""),
                    "output_path": str(output_path),
                }
                manifest_rows.append(row)
            except Exception as e:
                print(f"Job failed: {e}")
    return manifest_rows

def build_augmentation_jobs(records, total_images=24, variants=None, seed=42):
    rng = random.Random(seed)
    variants = variants or CFG.variants
    pool = [r for r in records if r.split in CFG.target_splits]
    if not pool:
        return []
    jobs = []
    for index in range(total_images):
        record = rng.choice(pool)
        variant = variants[index % len(variants)]
        out_path = generated_image_path(CFG.output_dir, record, variant, index)
        pair_path = comparison_image_path(CFG.output_dir, record, variant, index)
        jobs.append({"job_index": index, "source": load_source_image(record.path), "record": record, "variant": variant, "output_path": out_path, "comparison_path": pair_path, "seed": seed + index})
    return jobs

MANIFEST_FIELDS = [
    "split", "bucket", "source_context", "source_timeofday", "source_scene",
    "target_insertion", "target_timeofday", "original_path", "augmented_path",
    "comparison_path", "variant", "strength", "guidance_scale", "num_inference_steps",
    "generation_mode", "insert_bbox", "patch_bbox", "patch_debug_path", "person_bbox",
    "scale_corrected", "scale_ratios", "fallback_detection_used", "scale_correction_status", "seamless_clone_used",
    "fallback_alpha_paste", "fallback_alpha_used", "retry_attempts", "last_reject_reason",
    "reject_reason", "seed", "source_path", "label_path", "output_path",
]

def save_manifest(rows, output_dir):
    if not rows:
        print("No accepted rows; manifest not written.")
        return None
    path = Path(output_dir) / "manifest.csv"
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=MANIFEST_FIELDS, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)
    print(f"Saved manifest: {path}")
    return path

def _image_metrics(original, augmented):
    a = np.asarray(original.convert("RGB").resize((CFG.resolution, CFG.resolution)), dtype=np.float32)
    b = np.asarray(augmented.convert("RGB").resize((CFG.resolution, CFG.resolution)), dtype=np.float32)
    diff = a - b
    mse = float(np.mean(diff ** 2))
    mae = float(np.mean(np.abs(diff)))
    psnr = 99.0 if mse <= 1e-9 else float(20 * np.log10(255.0 / np.sqrt(mse)))
    brightness_delta = float(abs(a.mean() - b.mean()))
    contrast_delta = float(abs(a.std() - b.std()))
    hist_delta = 0.0
    for ch in range(3):
        ha, _ = np.histogram(a[..., ch], bins=32, range=(0, 255), density=True)
        hb, _ = np.histogram(b[..., ch], bins=32, range=(0, 255), density=True)
        hist_delta += float(np.abs(ha - hb).sum() / 2.0)
    return {"mse": mse, "mae": mae, "rmse": float(np.sqrt(mse)), "psnr": psnr, "brightness_delta": brightness_delta, "contrast_delta": contrast_delta, "histogram_distance": hist_delta / 3.0}

def compute_augmentation_metrics(manifest_rows, output_dir=CFG.output_dir):
    metrics_dir = Path(output_dir) / CFG.metrics_dirname
    metrics_dir.mkdir(parents=True, exist_ok=True)
    metric_path = metrics_dir / "augmentation_metrics.csv"
    metric_rows = []
    for row in manifest_rows:
        src_path = Path(row["source_path"])
        out_path = Path(row["output_path"])
        if not src_path.exists() or not out_path.exists():
            continue
        metrics = _image_metrics(load_source_image(src_path), load_source_image(out_path))
        metric_rows.append({"variant": row["variant"], "split": row["split"], "bucket": row["bucket"], "source_path": str(src_path), "output_path": str(out_path), **metrics})
    fields = ["variant", "split", "bucket", "source_path", "output_path", "mse", "mae", "rmse", "psnr", "brightness_delta", "contrast_delta", "histogram_distance"]
    with metric_path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(metric_rows)
    print(f"Saved augmentation metrics: {metric_path} ({len(metric_rows)} rows)")
    return metric_rows

def summarize_metric_rows(metric_rows, output_dir=CFG.output_dir):
    summary_path = Path(output_dir) / CFG.metrics_dirname / "augmentation_metrics_summary.csv"
    numeric = ["mse", "mae", "rmse", "psnr", "brightness_delta", "contrast_delta", "histogram_distance"]
    groups = {}
    for row in metric_rows:
        groups.setdefault(row["variant"], []).append(row)
    summary_rows = []
    for variant, rows in sorted(groups.items()):
        out = {"variant": variant, "count": len(rows)}
        for key in numeric:
            vals = [float(r[key]) for r in rows]
            out[f"{key}_mean"] = sum(vals) / max(1, len(vals))
        summary_rows.append(out)
    fields = ["variant", "count"] + [f"{key}_mean" for key in numeric]
    with summary_path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(summary_rows)
    print(f"Saved metrics summary: {summary_path}")
    return summary_rows

def augment_dataset(records, total_images=24, devices=None, write_metrics=True):
    jobs = build_augmentation_jobs(records, total_images=total_images)
    devices = devices or ([CFG.device] if torch.cuda.is_available() else ["cpu"])
    print(f"Using augmentation devices: {devices}")
    rows = run_augmentation_jobs(devices, jobs)
    save_manifest(rows, CFG.output_dir)
    if write_metrics and rows:
        metric_rows = compute_augmentation_metrics(rows, CFG.output_dir)
        summarize_metric_rows(metric_rows, CFG.output_dir)
    print(f"Generated {len(rows)} images in {CFG.output_dir}")
    return rows

In [ ]:
# Chuẩn bị job mẫu
sample_record = records[0] if records else None
if sample_record:
    sample_variant = "add_single_pedestrian"
    jobs = [{
        "job_index": 0,
        "source": load_source_image(sample_record.path),
        "record": sample_record,
        "variant": sample_variant,
        "output_path": generated_image_path(CFG.output_dir, sample_record, sample_variant, 0),
        "comparison_path": comparison_image_path(CFG.output_dir, sample_record, sample_variant, 0),
        "seed": 42
    }]
    
    print("Starting augmentation...")
    rows = run_augmentation_jobs([CFG.device], jobs)
    save_manifest(rows, CFG.output_dir)
    metric_rows = compute_augmentation_metrics(rows, CFG.output_dir)
    summarize_metric_rows(metric_rows, CFG.output_dir)
    print("Done! Check the output directory.")
else:
    print("No records found. Please check dataset path.")

In [ ]:
import shutil
from datetime import datetime

def archive_outputs(output_dir, archive_name=None):
    """Create a zip archive of all outputs."""
    output_path = Path(output_dir)
    if not output_path.exists():
        print(f"Output directory {output_dir} does not exist.")
        return None
    
    if archive_name is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        archive_name = f"sd35_citypersons_augmented_{timestamp}"
    
    archive_path = Path("/kaggle/working") / archive_name
    
    try:
        shutil.make_archive(str(archive_path), "zip", output_path.parent, output_path.name)
        archive_file = str(archive_path) + ".zip"
        size_mb = Path(archive_file).stat().st_size / (1024 * 1024)
        print(f"✓ Archived to {archive_file} ({size_mb:.2f} MB)")
        return archive_file
    except Exception as e:
        print(f"Failed to create archive: {e}")
        return None

# Create zip archive
zip_file = archive_outputs(str(CFG.output_dir))
if zip_file:
    print(f"Download: {Path(zip_file).name}")